# Feature Engineering, Model Optimization & Performance Comparison

**Objective:** Build a complete ML pipeline using the California Housing dataset - from data loading and preprocessing through model training, evaluation, comparison, and visualization.

---

## Step 1: Setup & Imports

All required libraries are imported below. We use:
- `pandas` / `numpy` - data handling
- `matplotlib` / `seaborn` - visualization
- `sklearn` - dataset, preprocessing, models, and metrics
- `joblib` - model persistence

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import warnings

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

print("All imports successful.")

---
## Step 2: Data Loading & Inspection

### About the Dataset

The **California Housing dataset** was derived from the 1990 U.S. census. Each row represents a **block group** (the smallest geographical unit for which the U.S. Census Bureau publishes sample data).

**Features:**
- `MedInc` - Median income in block group (in tens of thousands of USD)
- `HouseAge` - Median house age in block group
- `AveRooms` - Average number of rooms per household
- `AveBedrms` - Average number of bedrooms per household
- `Population` - Block group population
- `AveOccup` - Average number of household members
- `Latitude` - Block group latitude
- `Longitude` - Block group longitude

**Target:** `HousePrice` - Median house value in hundreds of thousands of USD (originally `MedHouseVal`, renamed for clarity).

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = pd.concat([housing.data, housing.target], axis=1)
df.rename(columns={"MedHouseVal": "HousePrice"}, inplace=True)

print("First 5 rows of the dataset:")
df.head()

In [ ]:
print("Dataset shape:", df.shape)
print()
print("Dataset info:")
df.info()

In [ ]:
print("Summary statistics:")
df.describe()

---
## Step 3: Data Preparation

### Why Feature Scaling is Critical

Feature scaling ensures all features contribute equally to the model. Algorithms like **Linear Regression** and **Ridge Regression** are sensitive to the magnitude of features because they compute distances and coefficients. Without scaling:
- Features with larger ranges (e.g., `Population`) dominate the loss function
- Coefficients become hard to interpret
- Gradient descent converges more slowly

**StandardScaler** transforms each feature to have **mean = 0** and **standard deviation = 1**, making them unitless and comparable.

Note: Tree-based models (Decision Tree) are invariant to feature scaling, but we scale for consistency across all models.

In [ ]:
X = df.drop("HousePrice", axis=1)
y = df["HousePrice"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
print("Before scaling - feature statistics:")
X.describe()

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

print("StandardScaler applied successfully.")

In [ ]:
print("After scaling - feature statistics (mean ~0, std ~1):")
X_scaled.describe()

---
## Step 4: Train-Test Split

We split the data into **80% training** and **20% testing** sets. A fixed `random_state=42` ensures reproducibility.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print("Training set:", X_train.shape, y_train.shape)
print("Testing set:", X_test.shape, y_test.shape)

---
## Step 5: Model Training

We train three regression models with different characteristics:

1. **Linear Regression** - Ordinary Least Squares baseline. Assumes linear relationship between features and target.
2. **Ridge Regression** - Linear model with L2 regularization (`alpha=1.0`) to penalize large coefficients and reduce overfitting.
3. **Decision Tree Regressor** - Non-parametric model (`max_depth=5`) that captures non-linear patterns and feature interactions.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Decision Tree": DecisionTreeRegressor(max_depth=5, random_state=42)
}

trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f"[OK] {name} trained successfully.")

print()
print("All models trained.")

---
## Step 6: Model Evaluation

We evaluate using two standard regression metrics:
- **RMSE (Root Mean Squared Error):** Measures average prediction error in the same units as the target. Lower is better.
- **R-squared (Coefficient of Determination):** Proportion of variance explained by the model. Higher is better (max = 1.0).

In [ ]:
results = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    results.append({
        "Model": name,
        "RMSE": rmse,
        "R-squared": r2
    })
    print(f"{name:25s} | RMSE: {rmse:.4f} | R-squared: {r2:.4f}")

In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE", ascending=True).reset_index(drop=True)

print("Model Performance Comparison (sorted by RMSE):")
results_df

### Interpretation

| Metric  | Meaning | Desired Direction |
|---------|---------|-------------------|
| **RMSE** | Average prediction error (in $100,000s of USD) | **Lower** is better |
| **R-squared** | Proportion of variance explained by the model | **Higher** is better (closer to 1.0) |

- A lower RMSE indicates the model's predictions are closer to actual values.
- A higher R-squared indicates the model captures more of the data's underlying patterns.
- The best model will have the **lowest RMSE** and **highest R-squared**.

---
## Step 7: Visualization

We create a comprehensive figure with three subplots:
1. **Actual vs. Predicted** scatter plot for the best model
2. **Model Performance Comparison** bar chart
3. **Residual Distribution** histogram with KDE for the best model

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
y_pred_best = best_model.predict(X_test)
residuals = y_test - y_pred_best

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Actual vs Predicted (best model)
axes[0].scatter(y_test, y_pred_best, alpha=0.5, edgecolors="none", s=20)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
             "r--", linewidth=2, label="Perfect Prediction")
axes[0].set_xlabel("Actual Prices")
axes[0].set_ylabel("Predicted Prices")
axes[0].set_title(f"Actual vs Predicted House Prices ({best_model_name})")
axes[0].legend()
axes[0].axis("equal")

# Plot 2: Model Performance Comparison
x_pos = np.arange(len(results_df))
width = 0.35

ax1 = axes[1]
ax1_twin = ax1.twinx()

ax1.bar(x_pos - width/2, results_df["RMSE"], width,
        label="RMSE", color="steelblue", edgecolor="black")
ax1_twin.bar(x_pos + width/2, results_df["R-squared"], width,
             label="R-squared", color="coral", edgecolor="black")

ax1.set_xlabel("Model")
ax1.set_ylabel("RMSE", color="steelblue")
ax1_twin.set_ylabel("R-squared", color="coral")
ax1.set_title("Model Performance Comparison")
ax1.set_xticks(x_pos)
ax1.set_xticklabels(results_df["Model"], rotation=15, ha="right")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_twin.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

# Plot 3: Residual Distribution (best model)
axes[2].hist(residuals, bins=40, density=True, alpha=0.6,
             color="steelblue", edgecolor="black", label="Residuals")
sns.kdeplot(residuals, ax=axes[2], color="darkred", linewidth=2, label="KDE")
axes[2].axvline(x=0, color="black", linestyle="--", linewidth=1, label="Zero Error")
axes[2].set_xlabel("Residual (Actual - Predicted)")
axes[2].set_ylabel("Density")
axes[2].set_title(f"Residual Distribution ({best_model_name})")
axes[2].legend()

plt.tight_layout()
plt.show()

---
## Step 8: Model Persistence

We save the **best-performing model** (lowest RMSE, highest R-squared) to disk using `joblib`. This allows reusing the trained model without retraining.

In [ ]:
best_rmse = results_df.iloc[0]["RMSE"]
best_r2 = results_df.iloc[0]["R-squared"]

joblib.dump(best_model, "./best_housing_model.pkl")
print(f"Best model saved as best_housing_model.pkl")
print()
print(f"Best model: {best_model_name}")
print(f"RMSE: {best_rmse:.4f}")
print(f"R-squared: {best_r2:.4f}")

In [ ]:
loaded_model = joblib.load("./best_housing_model.pkl")
y_pred_loaded = loaded_model.predict(X_test)
rmse_loaded = np.sqrt(mean_squared_error(y_test, y_pred_loaded))
print(f"Loaded model RMSE: {rmse_loaded:.4f}")
print(f"Integrity check passed: {abs(rmse_loaded - best_rmse) < 1e-10}")

### Why This Model Was Selected

The model with the **lowest RMSE** and **highest R-squared score** was selected as the best:
- It achieves the most accurate predictions with the smallest average error.
- It explains the largest proportion of variance in house prices.

This combination of low error and high explanatory power makes it the most reliable model for predicting California housing prices in this comparison.

---
## Step 9: Additional Insights

We provide three bonus analyses to deepen understanding of the data and models.

### 9.1 Correlation Heatmap

Shows pairwise Pearson correlations between all features and the target. Helps identify which features have the strongest linear relationship with house prices.

In [ ]:
corr_matrix = df.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, square=True, linewidths=0.5,
            cbar_kws={"shrink": 0.8})
plt.title("Feature Correlation Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print()
print("Top 3 features most correlated with HousePrice:")
print(corr_matrix["HousePrice"].drop("HousePrice").sort_values(ascending=False).head(3))

### 9.2 Feature Importance (Decision Tree)

The Decision Tree model provides built-in feature importance scores, indicating which features contribute most to splitting decisions.

In [ ]:
dt_model = trained_models["Decision Tree"]
importances = dt_model.feature_importances_
feature_names = X.columns

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(importance_df["Feature"], importance_df["Importance"],
         color="steelblue", edgecolor="black")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.title("Decision Tree - Feature Importance", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print()
print("Top 3 most important features (Decision Tree):")
print(importance_df.sort_values("Importance", ascending=False).head(3).to_string(index=False))

### 9.3 Prediction Demo

We show 5 random test samples with their actual vs. predicted house prices to get a qualitative sense of model performance.

In [ ]:
np.random.seed(42)
sample_indices = np.random.choice(len(X_test), size=5, replace=False)

demo_data = {
    "Sample": range(1, 6),
    "Actual Price": y_test.iloc[sample_indices].values,
    "Predicted Price": y_pred_best[sample_indices],
    "Error": y_test.iloc[sample_indices].values - y_pred_best[sample_indices],
    "Error %": ((y_test.iloc[sample_indices].values - y_pred_best[sample_indices])
                / y_test.iloc[sample_indices].values * 100)
}

demo_df = pd.DataFrame(demo_data)
print(f"Prediction Demo - Best Model: {best_model_name}")
print("=" * 70)
print(demo_df.to_string(index=False))
print("=" * 70)
print()
print("Note: Prices are in $100,000s of USD.")
print(f"Mean absolute % error across samples: {demo_df['Error %'].abs().mean():.2f}%")

---
## Summary & Key Learnings

### What was accomplished:

1. **Loaded and explored** the California Housing dataset (20,640 samples, 8 features)
2. **Preprocessed** the data with `StandardScaler` - verified mean ~ 0 and std ~ 1 post-scaling
3. **Split** into 80/20 train/test sets with reproducible randomness
4. **Trained 3 models:** Linear Regression (baseline), Ridge Regression (regularized), Decision Tree (non-linear)
5. **Evaluated** using RMSE (lower is better) and R-squared (higher is better)
6. **Visualized** model performance, actual vs. predicted, and residual distributions
7. **Persisted** the best model as `best_housing_model.pkl`
8. **Gained additional insights** through correlation heatmaps, feature importance analysis, and a prediction demo

### Key Takeaways:

- **Feature scaling** is essential for linear models but irrelevant for tree-based models.
- **Regularization** (Ridge) helps control overfitting and often improves generalization.
- **Decision Trees** can capture non-linear relationships but risk overfitting without depth constraints.
- **MedInc** (median income) was the strongest predictor of house prices in both correlation and feature importance analyses.

### Potential Improvements:

- Hyperparameter tuning via `GridSearchCV` or `RandomizedSearchCV`
- Ensemble methods: Random Forest, Gradient Boosting (XGBoost/LightGBM)
- Polynomial feature engineering for capturing interaction effects
- Outlier removal and target transformation (log-transform)

---
*End of notebook - AI_ML Task 2: Feature Engineering, Model Optimization & Performance Comparison*